# 資料處理

## 檢查版本

In [1]:
import sys, xarray as xr
print("Python exe:", sys.executable)
print("Engines:", xr.backends.list_engines())

Python exe: /home/jundian/miniconda3/envs/geospatial-neural-adapter/bin/python
Engines: {'netcdf4': <NetCDF4BackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using netCDF4 in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.NetCDF4BackendEntrypoint.html, 'h5netcdf': <H5netcdfBackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using h5netcdf in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.H5netcdfBackendEntrypoint.html, 'scipy': <ScipyBackendEntrypoint>
  Open netCDF files (.nc, .nc4, .cdf and .gz) using scipy in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html, 'store': <StoreBackendEntrypoint>
  Open AbstractDataStore instances in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html}


## 讀nc4

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm


def check_data_folder(folder: str) -> bool:
    return os.path.exists(folder) and os.path.isdir(folder)


def load_data(file_path: str) -> xr.Dataset:
    """
    Load data from a NetCDF file, trying netcdf4 then h5netcdf.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    # 優先 netcdf4
    try:
        return xr.open_dataset(file_path, engine="netcdf4")
    except Exception as e1:
        # 改試 h5netcdf
        try:
            return xr.open_dataset(file_path, engine="h5netcdf")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to open {file_path} with netcdf4 and h5netcdf.\n"
                f"e1: {e1}\n"
                f"e2: {e2}\n"
                "請確認這個環境有安裝 netCDF4 或 h5netcdf。"
            )


# ================= main program =================

data_folder = "nc4"
var_name = "T2M"  # 目標變數名稱（請確認檔內真的叫這個）

# 1. 檢查資料夾
if not check_data_folder(data_folder):
    raise FileNotFoundError(f"Data folder not found: {data_folder}")
print(f"Data folder found: {data_folder}")

# 2. 找出所有像 1980-01.nc4 的檔案
pattern = os.path.join(data_folder, "*.nc4")
file_list = sorted(glob.glob(pattern))

if len(file_list) == 0:
    raise FileNotFoundError(f"No nc4 files found with pattern: {pattern}")

print(f"Found {len(file_list)} files.")
print("First 5 files:", file_list[:5])

# 3. 用第一個檔案確認經緯度與變數存在，必要時自動偵測 var_name
with load_data(file_list[0]) as sample_data:
    print("Data variables in first file:")
    print(list(sample_data.data_vars))
    print("Coordinates:")
    print({k: sample_data[k].shape for k in sample_data.coords})

    # 找出所有長得像 (time, lat, lon) 的候選變數
    candidates = []
    for v in sample_data.data_vars:
        dims = set(sample_data[v].dims)
        if {"time", "lat", "lon"}.issubset(dims):
            candidates.append(v)

    # 如果原本設定的 var_name 不在，就試著自動改
    if var_name not in sample_data.data_vars:
        if len(candidates) == 1:
            auto_var = candidates[0]
            print(f"[Info] Variable '{var_name}' not found, auto-select '{auto_var}' as target.")
            var_name = auto_var
        else:
            raise KeyError(
                f"Variable '{var_name}' not found in file: {file_list[0]}\n"
                f"Available data_vars: {list(sample_data.data_vars)}\n"
                f"3D (time,lat,lon) candidates: {candidates}\n"
                "請將上方其中一個正確變數名稱填入 var_name。"
            )

    # 取經緯度
    if "lat" not in sample_data.coords or "lon" not in sample_data.coords:
        raise KeyError("lat/lon coordinates not found in sample file.")
    lat = sample_data["lat"].values
    lon = sample_data["lon"].values

nlat = lat.shape[0]
nlon = lon.shape[0]
print(f"Confirmed var_name = {var_name}")
print(f"Lat: {nlat}, Lon: {nlon}")


# 4. 逐檔讀入，累積到 list
data_list = []
time_list = []

for f in tqdm(file_list, desc="Combining"):
    with load_data(f) as ds:
        da = ds[var_name]  # (time, lat, lon)

        # 確保 lat/lon 一致（保險，可視情況註解）
        if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
            raise ValueError(f"Lat/Lon size mismatch in file: {f}")

        # 資料轉 float32，省記憶體
        data_list.append(da.values.astype(np.float32))

        if "time" not in ds:
            raise KeyError(f"'time' coordinate not found in file: {f}")

        # decode_cf 確保時間是真正 datetime
        t = xr.decode_cf(ds)["time"].values
        time_list.append(t.astype("datetime64[ns]"))

# 5. 串起來 → (ntot, lat, lon) & DatetimeIndex
combined = np.concatenate(data_list, axis=0)   # (ntot, nlat, nlon)
time_array = np.concatenate(time_list, axis=0) # (ntot,)

if combined.shape[0] != time_array.shape[0]:
    raise ValueError(
        f"time length ({time_array.shape[0]}) "
        f"!= data length ({combined.shape[0]})"
    )

# 依時間排序（通常已排序，這裡是保險）
sort_idx = np.argsort(time_array)
combined = combined[sort_idx]
time_array = time_array[sort_idx]

time_index = pd.to_datetime(time_array)

print(f"Combined data shape: {combined.shape}")
print(f"Time index: {time_index[0]} -> {time_index[-1]} (len={len(time_index)})")

# 6. 攤平成 cell × time
ntot, nlat, nlon = combined.shape
ncell = nlat * nlon

# (cell, time)
y_all = combined.reshape(ntot, ncell).T

# 建立每個 cell 的 (lon, lat)
lon_grid, lat_grid = np.meshgrid(lon, lat)
gg = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # (cell, 2)

print(f"y_all shape: {y_all.shape}  (cells x time)")
print(f"gg shape: {gg.shape}        (cells x [lon, lat])")


Data folder found: nc4
Found 548 files.
First 5 files: ['nc4/1980-01.nc4', 'nc4/1980-02.nc4', 'nc4/1980-03.nc4', 'nc4/1980-04.nc4', 'nc4/1980-05.nc4']
Data variables in first file:
['T2M', 'T2MDEW', 'Var_T2M', 'T2MWET']
Coordinates:
{'lon': (576,), 'time': (1,), 'lat': (361,)}
Confirmed var_name = T2M
Lat: 361, Lon: 576


Combining: 100%|██████████| 548/548 [00:14<00:00, 37.38it/s]


Combined data shape: (548, 361, 576)
Time index: 1980-01-01 00:30:00 -> 2025-08-01 00:30:00 (len=548)
y_all shape: (207936, 548)  (cells x time)
gg shape: (207936, 2)        (cells x [lon, lat])


# 限制範圍抽樣 in USA

In [3]:
# ===== 只看美國本土範圍，並從中抽樣 25 個格點 =====
import numpy as np

# gg: (ncell, 2)；第 0 欄是 lon，第 1 欄是 lat
lon_all = gg[:, 0].astype(float)
lat_all = gg[:, 1].astype(float)

# 如果經度是 0~360，轉成 -180~180（如果本來就是 -180~180，這個轉換也不會壞掉）
lon_all_180 = ((lon_all + 180) % 360) - 180

# 美國本土 48 州的大致範圍（可以再微調）
lon_min, lon_max = -125, -66
lat_min, lat_max = 24, 50

# 建立遮罩：只保留在美國本土範圍內的格點
mask_us_mainland = (
    (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
    (lat_all      >= lat_min) & (lat_all      <= lat_max)
)

idx_us_mainland = np.where(mask_us_mainland)[0]
print(f"美國本土範圍內的格點數量: {len(idx_us_mainland)}")

if len(idx_us_mainland) == 0:
    raise ValueError("在設定的美國本土範圍內沒有格點，請調整 lon_min/max 或 lat_min/max。")

# 只保留美國本土子集合
y_us = y_all[idx_us_mainland, :]  # (n_us, ntot)
coords_us = np.column_stack([lon_all_180[idx_us_mainland], lat_all[idx_us_mainland]])  # (n_us, 2)

print(f"美國本土子集合 y_us shape: {y_us.shape}")
print(f"美國本土子集合 coords_us shape: {coords_us.shape}")

# 從美國本土格點中隨機抽樣 25 個（如果不足 25，就全部使用）
n_sample = min(25, len(idx_us_mainland))
np.random.seed(42)  # 為了可重現
sample_idx_in_us = np.random.choice(len(idx_us_mainland), size=n_sample, replace=False)

# 這是「在原始格點編號中的 index」
sample_idx_global = idx_us_mainland[sample_idx_in_us]

# 抽樣後的 time series 與座標
y_sample = y_all[sample_idx_global, :]  # (n_sample, ntot)
coords_sample = np.column_stack([lon_all_180[sample_idx_global], lat_all[sample_idx_global]])  # (n_sample, 2)

# 如果之後畫圖想分開拿 lon / lat：
lon_us_sample = coords_sample[:, 0]
lat_us_sample = coords_sample[:, 1]

print(f"抽樣 {n_sample} 個美國本土格點")
print("前 5 個 sample index (global):", sample_idx_global[:5])
print("前 5 個 sample 座標 (lon, lat):")
for i in range(min(5, n_sample)):
    print(f"  #{i}: lon={lon_us_sample[i]:.2f}, lat={lat_us_sample[i]:.2f}")


美國本土範圍內的格點數量: 5035
美國本土子集合 y_us shape: (5035, 548)
美國本土子集合 coords_us shape: (5035, 2)
抽樣 25 個美國本土格點
前 5 個 sample index (global): [153335 159677 145263 156825 138939]
前 5 個 sample 座標 (lon, lat):
  #0: lon=-105.62, lat=43.00
  #1: lon=-101.88, lat=48.50
  #2: lon=-110.62, lat=36.00
  #3: lon=-84.38, lat=46.00
  #4: lon=-103.12, lat=30.50


## DLinaer

In [4]:
# ===== DLinear grid search on the 25 sampled US grid cells (compact output) =====
import numpy as np
import pandas as pd
from itertools import product

from darts import TimeSeries
from darts.models import DLinearModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("\n===== DLinear GRID SEARCH on 25 sampled US cells (compact output) =====")

# ============================================================
# 0) 參數集合（你只需要改這裡）
#    注意：set 會自動去重；順序不固定，我在後面用 sorted() 固定順序。
# ============================================================
INPUT_CHUNK_LENGTH_SET  = {48, 60, 72}
OUTPUT_CHUNK_LENGTH_SET = {12, 24}
KERNEL_SIZE_SET         = {13, 25, 37}
N_EPOCHS_SET            = {50, 100, 200}
USE_COVARIATES_SET      = {True, False}   # 用 / 不用 month covariate
REVIN_SET               = {False, True}   # use_reversible_instance_norm

RANDOM_STATE = 42

# ============================================================
# 0.1) 輸出控制（避免爆版）
# ============================================================
VERBOSE = 0   # 0=完全不印每組進度；1=每組都印；2=每10組印一次
TOPK = 5      # 最後顯示 TEST/VAL 各前 TOPK 名
SAVE_CSV = True
OUT_CSV = "dlinear_grid_results.csv"

# ============================================================
# 1) 建 target DataFrame：25 個格點 × T 時間點（原始單位）
# ============================================================
SAMPLE_SIZE = n_sample  # 預期是 25
T = y_sample.shape[1]

ts_df = pd.DataFrame(
    y_sample.T,  # (T, SAMPLE_SIZE)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T)
print("ts_df shape:", ts_df.shape)  # (T, SAMPLE_SIZE)

# ============================================================
# 2) month covariate（原始 month=1..12）
# ============================================================
month_values = time_index.month.astype("float32")
month_df = pd.DataFrame({"month": month_values}, index=time_index)
month_ts = TimeSeries.from_dataframe(month_df.astype("float32"))

# ============================================================
# 3) 切成 70 / 15 / 15（train / val / test）
# ============================================================
train_frac = 0.7
val_frac   = 0.15

cut_train = int(T * train_frac)
cut_val   = int(T * (train_frac + val_frac))

idx = ts_df.index
split_1_time = idx[cut_train]
split_2_time = idx[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)
train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
val_raw, test_raw  = tmp_raw.split_before(split_2_time)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

month_train, tmp_month = month_ts.split_before(split_1_time)
month_val, month_test  = tmp_month.split_before(split_2_time)

assert len(month_train) == len(train_raw)
assert len(month_val) == len(val_raw)
assert len(month_test) == len(test_raw)

# ============================================================
# 4) 用 train 統計量標準化
# ============================================================
train_df = ts_df.iloc[:cut_train]
mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)

ts_df_scaled = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
val_ts, test_ts  = tmp_ts.split_before(split_2_time)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)

vals_all      = ts_df.to_numpy(dtype=np.float32)
val_true_raw  = vals_all[cut_train:cut_val, :]
test_true_raw = vals_all[cut_val:, :]

def inverse_scale(x: np.ndarray) -> np.ndarray:
    return x * std_vec.values + mean_vec.values

print("\n[Scaled TimeSeries lengths]")
print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 用於 TEST 訓練：train+val
train_val_ts    = train_ts.concatenate(val_ts)
month_train_val = month_train.concatenate(month_val)

# ============================================================
# 5) Grid search
# ============================================================
def _ts_to_2d(ts: TimeSeries) -> np.ndarray:
    arr = np.asarray(ts.all_values(copy=False))
    if arr.ndim == 3:
        arr = arr[..., 0]  # (T, C)
    return arr

def eval_one_setting(input_len, output_len, kernel_size, n_epochs, use_cov, use_revin):
    # --- VAL ---
    model_val = DLinearModel(
        input_chunk_length=input_len,
        output_chunk_length=output_len,
        kernel_size=kernel_size,
        n_epochs=n_epochs,
        random_state=RANDOM_STATE,
        use_reversible_instance_norm=use_revin,
    )

    fit_kwargs_val = {"series": train_ts}
    if use_cov:
        fit_kwargs_val["past_covariates"] = month_train
        fit_kwargs_val["future_covariates"] = month_train
    model_val.fit(**fit_kwargs_val)

    pred_kwargs_val = {"n": T_val}
    if use_cov:
        pred_kwargs_val["past_covariates"] = month_ts
        pred_kwargs_val["future_covariates"] = month_ts
    pred_val = model_val.predict(**pred_kwargs_val)

    pred_val_scaled = _ts_to_2d(pred_val)
    pred_val_raw = inverse_scale(pred_val_scaled)

    # --- TEST ---
    model_test = DLinearModel(
        input_chunk_length=input_len,
        output_chunk_length=output_len,
        kernel_size=kernel_size,
        n_epochs=n_epochs,
        random_state=RANDOM_STATE,
        use_reversible_instance_norm=use_revin,
    )

    fit_kwargs_test = {"series": train_val_ts}
    if use_cov:
        fit_kwargs_test["past_covariates"] = month_train_val
        fit_kwargs_test["future_covariates"] = month_train_val
    model_test.fit(**fit_kwargs_test)

    pred_kwargs_test = {"n": T_test}
    if use_cov:
        pred_kwargs_test["past_covariates"] = month_ts
        pred_kwargs_test["future_covariates"] = month_ts
    pred_test = model_test.predict(**pred_kwargs_test)

    pred_test_scaled = _ts_to_2d(pred_test)
    pred_test_raw = inverse_scale(pred_test_scaled)

    # --- metrics ---
    y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
    y_val_pred  = pred_val_raw.reshape(-1, SAMPLE_SIZE)
    y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
    y_test_pred = pred_test_raw.reshape(-1, SAMPLE_SIZE)

    mse_val  = mean_squared_error(y_val_true,  y_val_pred)
    mae_val  = mean_absolute_error(y_val_true, y_val_pred)
    rmse_val = float(np.sqrt(mse_val))
    r2_val   = r2_score(y_val_true, y_val_pred)

    mse_test  = mean_squared_error(y_test_true,  y_test_pred)
    mae_test  = mean_absolute_error(y_test_true, y_test_pred)
    rmse_test = float(np.sqrt(mse_test))
    r2_test   = r2_score(y_test_true, y_test_pred)

    return (rmse_val, mse_val, mae_val, r2_val), (rmse_test, mse_test, mae_test, r2_test)

grid = list(product(
    sorted(INPUT_CHUNK_LENGTH_SET),
    sorted(OUTPUT_CHUNK_LENGTH_SET),
    sorted(KERNEL_SIZE_SET),
    sorted(N_EPOCHS_SET),
    sorted(USE_COVARIATES_SET),
    sorted(REVIN_SET),
))

results = []
total = len(grid)
print(f"\nTotal settings to run: {total}")

for i, (in_len, out_len, ksz, nep, use_cov, use_revin) in enumerate(grid, 1):
    if VERBOSE == 1:
        print(f"[{i}/{total}] input={in_len}, output={out_len}, kernel={ksz}, epochs={nep}, cov={use_cov}, revin={use_revin}")
    elif VERBOSE == 2 and (i % 10 == 0 or i == 1 or i == total):
        print(f"[{i}/{total}] ... running")

    (rmse_v, mse_v, mae_v, r2_v), (rmse_t, mse_t, mae_t, r2_t) = eval_one_setting(
        input_len=in_len,
        output_len=out_len,
        kernel_size=ksz,
        n_epochs=nep,
        use_cov=use_cov,
        use_revin=use_revin,
    )

    results.append({
        "input_chunk_length": in_len,
        "output_chunk_length": out_len,
        "kernel_size": ksz,
        "n_epochs": nep,
        "use_covariates": use_cov,
        "use_revin": use_revin,
        "Split": "VAL",
        "RMSE": rmse_v,
        "MSE": mse_v,
        "MAE": mae_v,
        "R2": r2_v,
    })
    results.append({
        "input_chunk_length": in_len,
        "output_chunk_length": out_len,
        "kernel_size": ksz,
        "n_epochs": nep,
        "use_covariates": use_cov,
        "use_revin": use_revin,
        "Split": "TEST",
        "RMSE": rmse_t,
        "MSE": mse_t,
        "MAE": mae_t,
        "R2": r2_t,
    })

results_df = pd.DataFrame(results)

# ============================================================
# 6) 只顯示少量結果 + 可選存檔
# ============================================================
if SAVE_CSV:
    results_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\nSaved full results to: {OUT_CSV}")

# 只顯示 TEST / VAL 各前 TOPK 名（依 RMSE 由小到大）
test_top = results_df[results_df["Split"] == "TEST"].sort_values("RMSE").head(TOPK)
val_top  = results_df[results_df["Split"] == "VAL"].sort_values("RMSE").head(TOPK)

# 欄位裁切：只留你關心的設定 + 指標
cols = [
    "input_chunk_length", "output_chunk_length", "kernel_size", "n_epochs",
    "use_covariates", "use_revin", "Split", "RMSE", "MSE", "MAE", "R2"
]

print(f"\n===== TOP {TOPK} (TEST, sorted by RMSE) =====")
print(test_top[cols].to_string(index=False))

print(f"\n===== TOP {TOPK} (VAL, sorted by RMSE) =====")
print(val_top[cols].to_string(index=False))



/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | t


===== DLinear GRID SEARCH on 25 sampled US cells (compact output) =====
Total time steps: 548
ts_df shape: (548, 25)
Train len (raw): 383, Val len: 82, Test len: 83

[Scaled TimeSeries lengths]
T_train = 383 T_val = 82 T_test = 83

Total settings to run: 216
Epoch 49: 100%|██████████| 11/11 [00:00<00:00, 116.29it/s, train_loss=0.0226]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 11/11 [00:00<00:00, 113.41it/s, train_loss=0.0226]


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


...

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 199: 100%|██████████| 12/12 [00:00<00:00, -24.65it/s, train_loss=0.00746]

`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 148.04it/s]

Saved full results to: dlinear_grid_results.csv

===== TOP 5 (TEST, sorted by RMSE) =====
 input_chunk_length  output_chunk_length  kernel_size  n_epochs  use_covariates  use_revin Split     RMSE      MSE      MAE       R2
                 48                   24           25        50           False       True  TEST 1.748973 3.058905 1.284402 0.932516
                 48                   24           37        50           False       True  TEST 1.752748 3.072126 1.283533 0.933755
                 48                   24           13        50           False       True  TEST 1.774283 3.148081 1.305830 0.929505
                 72                   12           13        50            True       True  TEST 1.781119 3.172386 1.275100 0.928445
                 72                   12           25        50            True       True  TEST 1.781777 3.174730 1.288337 0.924235

===== TOP 5 (VAL, sorted by RMSE) ====

### Recording
|  n_epochs | Split |   RMSE   |    MSE    |   MAE    |    R²    |
|:-------:|:-----:|:--------:|:---------:|:--------:|:--------:|
| 50 |  VAL  | 2.835351 | 8.039214  | 2.023799 | 0.837339 |
| 50 | TEST  | 1.810842 | 3.279148  | 1.348629 | 0.912932 |
| 100 |  VAL  | 2.983060 | 8.898644  | 2.102170 | 0.822471 |
| 100 | TEST  | 1.966121 | 3.865632  | 1.441832 | 0.900918 |
| 200 |  VAL  | 3.083795 | 9.509793  | 2.170602 | 0.803638 |
| 200 | TEST  | 2.394551 | 5.733873  | 1.750595 | 0.874610 |


### DLINEAR Results — 25 Sampled US Grid Cells

**Data Summary**  
- Total time steps: **548**  
- Grid cells: **25**  
- Train length: **383**  
- Validation length: **82**  
- Test length: **83**

---

### Performance Summary

| Model   | Split |   RMSE   |    MSE    |    MAE   |    R²     |
|:-------:|:-----:|:--------:|:---------:|:--------:|:---------:|
| DLINEAR | VAL   | 2.171805 | 4.716735  | 1.541092 | 0.891085  |
| DLINEAR | TEST  | 1.913623 | 3.661953  | 1.422750 | 0.907430  |


